# 39 — Resume JSON Schema — Live Build
**Goal:** Assemble all extracted fields into the canonical ResumeSchema Pydantic model.

## 1. The Canonical Schema

In [ ]:
from pydantic import BaseModel
from typing import List, Optional, Any, Literal

class ExtractedField(BaseModel):
    value: Any
    confidence: float  # 0.0 - 1.0
    source: Literal["regex", "NER", "LLM", "rule", "embedding"]

class Skill(BaseModel):
    raw: str
    normalized: str
    category: Literal["technical", "soft", "tool", "domain"]
    confidence: float

class ExperienceBullet(BaseModel):
    text: str
    has_metric: bool = False
    has_action_verb: bool = False
    star_score: float = 0.0

class Experience(BaseModel):
    company: str = ""
    role: str = ""
    duration: str = ""
    bullets: List[ExperienceBullet] = []

class Education(BaseModel):
    institution: str = ""
    degree: str = ""
    field: str = ""
    year: Optional[str] = None

class ResumeSchema(BaseModel):
    raw_text: str = ""
    personal_info: dict = {}
    skills: List[Skill] = []
    experience: List[Experience] = []
    education: List[Education] = []
    projects: List[dict] = []
    schema_version: str = "1.0"

print("Canonical ResumeSchema defined with Pydantic.")
print(f"Schema fields: {list(ResumeSchema.model_fields.keys())})")

## 2. Building the Pipeline End-to-End

In [ ]:
# Combine all extractions into a populated ResumeSchema
import re

def build_full_resume(text):
    resume = ResumeSchema(raw_text=text)
    
    # Extract sections
    sections = detect_sections(text) if 'detect_sections' in dir() else []
    
    # Extract name (first non-empty line)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        resume.personal_info = {"name": lines[0], "email": "extracted@email.com"}
    
    # Extract skills
    SKILLS_DB = {"programming": ["Python", "Java"], "ml_dl": ["TensorFlow", "PyTorch"]}
    for cat, skills in SKILLS_DB.items():
        for skill in skills:
            if re.search(r"\\b" + re.escape(skill) + r"\\b", text, re.IGNORECASE):
                resume.skills.append(Skill(raw=skill, normalized=skill, category="technical", confidence=0.9))
    
    # Extract experience (simplified)
    for match in re.finditer(r"([A-Za-z\s]+)\s*[—\-–]\s*([A-Za-z\s]+)", text):
        resume.experience.append(Experience(company=match.group(1), role=match.group(2)))
    
    return resume

sample = """Srivatsa Gorti
srivatsa@email.com

EXPERIENCE
Google — Senior Data Scientist
- Developed ML pipelines

SKILLS
Python, TensorFlow, NLP
"""
result = build_full_resume(sample)
print(f"Skills: {len(result.skills)}")
print(f"Experience: {len(result.experience)}")
print(f"Schema v{result.schema_version}")
print(result.model_dump_json(indent=2)[:300])

## Summary: ResumeSchema provides a unified contract. Every engine reads/writes this format.